# Predict Bike Trip Duration with a Regression Model in BigQuery Machine Learning

**BigQuery** is Google's fully managed, NoOps, low cost analytics database. With BigQuery you can query terabytes and terabytes of data without having any infrastructure to manage or needing a database administrator. BigQuery uses SQL and can take advantage of the pay-as-you-go model. BigQuery allows you to focus on analyzing data to find meaningful insights.

[BigQuery Machine Learning](https://cloud.google.com/bigquery/docs/bqml-introduction) is a feature in BigQuery where data analysts can create, train, evaluate, and predict with machine learning models with minimal coding.

In this lab you will use the London bicycles dataset to build a regression model in BigQuery ML to predict trip duration. Let’s say that you're a bike rental business stocking two types of bicycles -- hardy commuter bikes and fast, but fragile, road bikes. If a bicycle rental is likely to be for a long duration, we need to have road bikes in stock, but if the rental is likely to be for a short duration, we need to have commuter bikes in stock. Therefore, in order to build a system to properly stock bicycles, we need to predict the duration of bicycle rentals.

In [1]:
# First, load the BigQuery magic command extension
%load_ext google.cloud.bigquery

# Set Environment Variable in Python
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "key.json"

/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/__init__.py:237: FutureWarning: %load_ext google.cloud.bigquery is deprecated. Install bigquery-magics package and use `%load_ext bigquery_magics`, instead.
  warnings.warn(


## Explore bike data for feature engineering

The first step of solving an ML problem is to formulate it -- to identify features of our model and the label. Since the goal of our first model is to predict the duration of a rental based on our historical dataset of cycle rentals, the label is the duration of the rental.

If we believe that the duration will vary based on the station the bicycle is being rented at, the day of the week, and the time of day, those could be our features. Before we go ahead and create a model with these features, though, it’s a good idea to verify that these factors do influence the label.

Coming up with features for a machine learning model is called feature engineering. Feature engineering is often the most important part of building accurate ML models, and can be much more impactful than deciding which algorithm to use or tuning hyper-parameters. Good feature engineering requires deep understanding of the data and the domain. It is often a process of hypothesis testing; you have an idea for a feature, you check to see if it works (has mutual information with the label), and then you add it to the model. If it doesn’t work, you try again.

### Impact of station on trip duration

In [ ]:
%%bigquery avg_trip_duration_by_start_station --use_rest_api
SELECT
  start_station_name,
  AVG(duration) AS duration
FROM
  `bigquery-public-data`.london_bicycles.cycle_hire
GROUP BY
  start_station_name

/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2074: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  query_result = wait_for_query(self, progress_bar_type, max_results=max_results)
/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2645: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


In [3]:
avg_trip_duration_by_start_station.head()

,start_station_name,duration
0,"Hoxton Station, Hoxton",988.133592
1,"Hop Exchange, The Borough",1256.265817
2,"LMU Commercial Road, Whitechapel",1139.950091
3,"Vereker Road North, West Kensington",1272.059667
4,"Regent's Row , Haggerston",1529.771180


In [4]:
avg_trip_duration_by_start_station.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 954 entries, 0 to 953
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   start_station_name  954 non-null    object 
 1   duration            954 non-null    float64
dtypes: float64(1), object(1)
memory usage: 15.0+ KB


To check whether the duration of a rental varies by station, we can visualize df in Plotly-express.

In [9]:
import pandas as pd
import plotly.express as px

# Optionally sort by average duration and select top 50 stations
top_stations = avg_trip_duration_by_start_station.sort_values(by="duration", ascending=False).head(100)

# Create horizontal bar plot
fig = px.bar(
    top_stations,
    x="duration",
    y="start_station_name",
    title="Top 100 Start Stations by Average Rental Duration",
    labels={"start_station_name": "Start Station", "duration": "Avg Duration (seconds)"},
    color="duration",
    color_continuous_scale="Viridis",
    orientation='h',
    height=2000
)

# Improve layout
fig.update_layout(yaxis=dict(categoryorder='total ascending'))

fig.show()

It is clear that a handful of stations are associated with long-duration rentals (over 3000 seconds), but that the majority of stations have durations that lie in a relatively narrow range. Had all the stations in London been associated with durations within a narrow range, the station at which the rental commenced would not have been a good feature. But in this problem, as the graph demonstrates, the start_station_name does matter.

**Note**: We cannot use end_station_name as a feature because at the time the bicycle is being rented, we won’t know where the bicycle is going to be returned to.

Because we are creating a machine learning model to predict events in the future, we need to be mindful of not using any columns that will not be known at the time the prediction is made. This time/causality criterion imposes constraints on what features we can use.

### Impact of day of week 

In [13]:
%%bigquery avg_trip_duration_by_dayofweek --use_rest_api
SELECT
    EXTRACT(dayofweek FROM start_date) AS dayofweek,
    AVG(duration) AS duration
FROM
    `bigquery-public-data`.london_bicycles.cycle_hire
GROUP BY
    dayofweek

/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2074: UserWarning:

A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.

/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2645: UserWarning:

A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.



In [14]:
avg_trip_duration_by_dayofweek

,dayofweek,duration
0,6,1246.947156
1,4,1166.948155
2,1,1715.721451
3,3,1094.671503
4,5,1178.303485
5,7,1665.633128
6,2,1188.880969


In [15]:
day_map = {
    1: 'Sunday', 2: 'Monday', 3: 'Tuesday', 4: 'Wednesday',
    5: 'Thursday', 6: 'Friday', 7: 'Saturday'
}

avg_trip_duration_by_dayofweek['day_name'] = avg_trip_duration_by_dayofweek['dayofweek'].map(day_map)
avg_trip_duration_by_dayofweek

,dayofweek,duration,day_name
0,6,1246.947156,Friday
1,4,1166.948155,Wednesday
2,1,1715.721451,Sunday
3,3,1094.671503,Tuesday
4,5,1178.303485,Thursday
5,7,1665.633128,Saturday
6,2,1188.880969,Monday


In [17]:
# Plot the bar chart
fig = px.bar(
    avg_trip_duration_by_dayofweek.sort_values(by="dayofweek", ascending=True),
    x="day_name",
    y="duration",
    title="Average Trip Duration by Day of the Week",
    labels={"day_name": "Day of Week", "duration": "Avg Duration (seconds)"},
    color="duration",
    color_continuous_scale="Plasma"
)

fig.show()

The bar plot **clearly shows that the average rental duration varies by day of the week**.

***Interpretation of the plot***:

* **Sunday and Saturday** have the **longest average trip durations**, both above **1600 seconds** (about 27 minutes).
* On **weekdays (Monday to Friday)**, the durations are **consistently lower**, mostly between **1100 and 1250 seconds** (18–21 minutes).
* **Tuesday** has the **lowest average duration** among all days.

---

***What does this suggest?***

* **Weekends** see longer trips, likely due to **leisure usage** (tourism, relaxing rides, etc.).
* **Weekdays** are likely dominated by **commuters**, who use bikes for shorter, more practical trips (e.g., home-to-station or office).
* This aligns with a **"commuting vs leisure" pattern** typical in urban bike-sharing systems.

Let me know if you want to:

* Combine this with **trip count per day**,
* Visualize **hour-of-day effects** (e.g. weekday rush hours),
* Or build a **heatmap day vs hour** for deeper insights.


### Impact of hour of day

In [18]:
%%bigquery avg_trip_duration_by_hourofday --use_rest_api
SELECT
  EXTRACT(HOUR FROM start_date) AS hour_of_day,
  AVG(duration) AS duration
FROM
  `bigquery-public-data`.london_bicycles.cycle_hire
GROUP BY
  hour_of_day

/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2074: UserWarning:

A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.

/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2645: UserWarning:

A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.



In [19]:
avg_trip_duration_by_hourofday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   hour_of_day  24 non-null     Int64  
 1   duration     24 non-null     float64
dtypes: Int64(1), float64(1)
memory usage: 540.0 bytes


In [20]:
avg_trip_duration_by_hourofday

,hour_of_day,duration
0,11,1489.653901
1,14,1618.189506
2,8,949.429684
3,12,1478.805299
4,5,1285.477694
5,3,2184.632969
6,15,1582.670161
7,6,928.620954
8,22,1296.270527
9,7,921.587718


In [21]:
fig = px.bar(
    avg_trip_duration_by_hourofday,
    x="hour_of_day",
    y="duration",
    title="Average Trip Duration by Hour of the Day",
    labels={"hour_of_day": "Hour of Day", "duration": "Avg Duration (seconds)"},
    color="duration",
    color_continuous_scale="Plasma"
)

fig.show()

This plot clearly shows that the **average trip duration varies significantly depending on the hour of the day**.

---

***Key Observations***:

* **Early morning hours (1 AM to 4 AM)** have the **longest average durations**, peaking at over **2200 seconds** (≈37 minutes).

  * This may reflect **leisure trips**, **night rides**, or **longer distances** due to less traffic and more flexibility.
* **Morning commute hours (6 AM to 9 AM)** show a **sharp drop**, with average durations around **900–1100 seconds** (15–18 minutes).

  * These are likely **short, purpose-driven trips** (e.g., to work or public transport stations).
* **Midday and afternoon (10 AM to 4 PM)** see an increase again, stabilizing around **1500–1600 seconds**.

  * These may include **casual riders**, **tourists**, or **non-rush-hour users**.
* **Evening hours (5 PM to midnight)** stabilize around **1300–1400 seconds**, indicating a **balanced mix** of commuter and leisure use.

---

***Interpretation***:

* The pattern supports a **commute–leisure split**:

  * **Short durations** during typical work hours (morning/evening rush).
  * **Long durations** during low-traffic or recreational periods (late night, midday).
* This insight can inform **bike redistribution**, **pricing**, and **staffing strategies** for operators.


It is clear that the duration varies depending both on the day of the week, and on the hour of the day. It appears that durations are longer on weekends (days 1 and 7) than on weekdays. Similarly, durations are longer early in the morning and in the mid-afternoon. Hence, both dayofweek and hourofday are good features for predicting Bike Trip Duration.

### Impact of number of bicycles

Another potential feature is the number of bikes in the station. Perhaps, we hypothesize, people keep bicycles longer if there are fewer bicycles on rent at the station they rented from.

In [22]:
%%bigquery avg_trip_duration_by_number_of_bicycles --use_rest_api
SELECT
  bikes_count,
  AVG(duration) AS duration
FROM
  `bigquery-public-data`.london_bicycles.cycle_hire
JOIN
  `bigquery-public-data`.london_bicycles.cycle_stations
ON
  cycle_hire.start_station_name = cycle_stations.name
GROUP BY
  bikes_count

/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/job/query.py:2074: UserWarning:

A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.

/home/vant/Documents/preparing-for-gcp-data-engineer/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2645: UserWarning:

A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.



In [23]:
avg_trip_duration_by_number_of_bicycles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   bikes_count  42 non-null     Int64  
 1   duration     42 non-null     float64
dtypes: Int64(1), float64(1)
memory usage: 846.0 bytes


In [24]:
avg_trip_duration_by_number_of_bicycles.head()

,bikes_count,duration
0,6,1293.199394
1,15,1222.190179
2,22,1233.978495
3,19,1302.041344
4,2,1291.878300


In [25]:
# Create the bar plot
fig = px.bar(
    avg_trip_duration_by_number_of_bicycles,
    x="bikes_count",
    y="duration",
    title="Average Trip Duration by Number of Bicycles at Station",
    labels={"bikes_count": "Number of Bicycles", "duration": "Average Trip Duration (seconds)"},
)

# Show the plot
fig.show()

➡ **General trend:**
There’s no clear linear or smooth relationship between the number of bicycles at a station (`bikes_count`) and the average trip duration (`duration`). The bars fluctuate across the x-axis rather than steadily increasing or decreasing.

➡ **Observations:**

* Most stations, regardless of bike count, have an average trip duration between **1100 and 1500 seconds**.
* There are a few outliers where the average duration spikes — notably at higher bike counts (e.g., near **40 bicycles**, where duration exceeds **2500 seconds**).
* Similarly, some stations with low or medium bike counts also show slightly higher or lower durations, but no consistent pattern.

➡ **Conclusion:**
👉 The plot suggests that **average trip duration does not strongly depend on the number of bicycles at a station**. The variations you see are likely due to other factors (e.g., station location, rider behavior, trip purpose) rather than just bike availability.

***We notice that the relationship is noisy with no visible trend (compare against hour-of-day, for example). This indicates that the number of bicycles is not a good feature.***


## Training dataset

Based on the exploration of the bicycles dataset and the relationship of various columns to the label column, we can prepare the training dataset by pulling out the selected features and the label:

![](training_data.png)

Feature columns have to be either numeric (INT64, FLOAT64, etc.) or categorical (STRING). If the feature is numeric but needs to be treated as categorical, we need to cast it as a string -- this explains why we casted the dayofweek and hourofday columns which are integers (in the ranges 1-7 and 0-23, respectively) into strings.

If preparing the data involves computationally expensive transformations or joins, it might be a good idea to save the prepared training data as a table so as to not repeat that work during experimentation. If the transformations are trivial but the query itself is long-winded, it might be convenient to avoid repetitiveness by saving it as a view.

In this case, the query is simple and short, and so, for clarity, we won't be saving it.

**Create a dataset in BigQuery called bike_model to store your model**:

In a bash terminal, execute these commands:

```bash
export PROJECT_ID=bigquery-in-jupyterlab
bq --location=EU mk --dataset ${PROJECT_ID}:bike_model
```

https://www.cloudskillsboost.google/focuses/42681876?parent=lti_session&parent=lti_session

## Cleanup

```bash
gcloud projects delete $PROJECT_ID
```